In [1]:
import warnings
import numpy as np
import optuna

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Suppress verbose Optuna logs to keep the terminal output clean
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Precision Settings
NP_DTYPE = np.float32

# ============================================================
# 1. Reproducibility & Data Loading
# ============================================================
base_seed = 2026
np.random.seed(base_seed)

# Load Clean Reference
data_clean = np.load("lorenz63_noise_0.npy", allow_pickle=True).item()
l63 = data_clean["data"]
if l63.shape[0] == 3:  # Force (T, d) layout standard
    l63 = l63.T
l63 = l63.astype(NP_DTYPE)

# Load Noisy Dataset (30% global measurement noise added)
data_noisy = np.load("lorenz63_noise_30.npy", allow_pickle=True).item()
X_noisy_dataset = data_noisy["data"]
if X_noisy_dataset.shape[0] == 3:
    X_noisy_dataset = X_noisy_dataset.T
X_noisy_dataset = X_noisy_dataset.astype(NP_DTYPE)

# Match exact splitting boundary points from NG-RC script
warmup_len, train_len, val_len, test_len = 100, 3900, 500, 500

X_train = X_noisy_dataset[warmup_len : warmup_len + train_len]
X_val   = X_noisy_dataset[warmup_len + train_len : warmup_len + train_len + val_len]
X_test  = X_noisy_dataset[warmup_len + train_len + val_len : warmup_len + train_len + val_len + test_len]

X_val_true  = l63[warmup_len + train_len : warmup_len + train_len + val_len]
X_test_true = l63[warmup_len + train_len + val_len : warmup_len + train_len + val_len + test_len]

# Structural Parameters
d = 3
horizons = [25, 50, 75, 100]

# ============================================================
# 2. Echo State Network Class Definition
# ============================================================
class EchoStateNetwork:
    def __init__(self, n_input, n_reservoir, spectral_radius, input_scaling, leak_factor, connectivity, seed):
        self.rng = np.random.RandomState(seed)
        self.n_input = n_input
        self.n_reservoir = n_reservoir
        self.leak_factor = np.float32(leak_factor)
        
        # Initialize input weights
        self.W_in = self.rng.uniform(-input_scaling, input_scaling, (n_reservoir, n_input)).astype(NP_DTYPE)
        
        # Initialize internal reservoir weights
        W = self.rng.uniform(-1.0, 1.0, (n_reservoir, n_reservoir)).astype(NP_DTYPE)
        mask = (self.rng.rand(n_reservoir, n_reservoir) < connectivity)
        W *= mask
        
        # Scale by spectral radius
        eigvals = np.abs(np.linalg.eigvals(W))
        maxeig = np.max(eigvals)
        if maxeig == 0:
            maxeig = np.float32(1.0)
        self.W = ((W / maxeig) * spectral_radius).astype(NP_DTYPE)
        
        self.W_out = None

    def _generate_states(self, U):
        """Standard reservoir state collection loop"""
        T = U.shape[0]
        states = np.zeros((T, self.n_reservoir), dtype=NP_DTYPE)
        x = np.zeros(self.n_reservoir, dtype=NP_DTYPE)
        
        for t in range(T):
            x = (1.0 - self.leak_factor) * x + self.leak_factor * np.tanh(
                self.W @ x + self.W_in @ U[t]
            )
            states[t] = x
        return states

    def fit(self, X_input, ridge_param):
        """Train output weights mapping state matrix to next time step target"""
        O_train = self._generate_states(X_input[:-1])
        Y_train = X_input[1:]  # Next-step targets
        
        C = O_train.T @ O_train
        regularizer = ridge_param * np.eye(C.shape[0], dtype=NP_DTYPE)
        
        try:
            self.W_out = np.linalg.solve(C + regularizer, O_train.T @ Y_train)
        except np.linalg.LinAlgError:
            self.W_out = np.linalg.pinv(C + regularizer) @ O_train.T @ Y_train
        return self

    def forecast_autonomous(self, X_history, u0, horizon):
        """Warm up the reservoir using history sequence, then generate free-running predictions"""
        # Step 1: Evolve reservoir state over the available sequence history window
        x = np.zeros(self.n_reservoir, dtype=NP_DTYPE)
        for t in range(X_history.shape[0]):
            x = (1.0 - self.leak_factor) * x + self.leak_factor * np.tanh(
                self.W @ x + self.W_in @ X_history[t]
            )
            
        # Step 2: Autonomous free-running closed-loop forecast
        preds = np.zeros((horizon, self.n_input), dtype=NP_DTYPE)
        u = u0.copy()
        
        for t in range(horizon):
            x = (1.0 - self.leak_factor) * x + self.leak_factor * np.tanh(
                self.W @ x + self.W_in @ u
            )
            y = x @ self.W_out
            preds[t] = y
            u = y  # Feed output prediction back as next input step
            
        return preds

# ============================================================
# 3. Core Evaluation Pipeline Function (Overlapping Sliding Windows)
# ============================================================
def evaluate_esn_model(esn_model, X_true_target, X_history, horizon_max, stride, h_list, washout_len=100):
    horizon_rmses = {h: [] for h in h_list}
    total_len = len(X_true_target)
    
    # Slide through set with a stride of 10 steps for exact fair layout evaluation
    for start_idx in range(washout_len, total_len - horizon_max + 1, stride):
        y_true_window = X_true_target[start_idx : start_idx + horizon_max]
        
        # History segment used to wash out and settle reservoir nodes
        X_hist_window = X_history[start_idx - washout_len : start_idx]
        u0 = X_history[start_idx]
        
        # Predict direct absolute coordinate state trajectory
        predictions_np = esn_model.forecast_autonomous(X_hist_window, u0, horizon=horizon_max)
        
        for h in h_list:
            rmse = np.sqrt(np.mean((predictions_np[:h] - y_true_window[:h]) ** 2))
            horizon_rmses[h].append(rmse)
            
    return {h: np.mean(horizon_rmses[h]) for h in h_list}

def objective(trial):
    # Search spaces setup matching high-performance ESN exploration boundaries
    n_reservoir = trial.suggest_categorical("n_reservoir", [64, 128, 256, 512])
    spectral_radius = trial.suggest_float("spectral_radius", 0.1, 1.5)
    input_scaling = trial.suggest_float("input_scaling", 0.01, 1.0, log=True)
    leak_factor = trial.suggest_float("leak_factor", 0.1, 1.0)
    connectivity = trial.suggest_float("connectivity", 0.01, 0.3)
    ridge_param = trial.suggest_categorical("ridge_param", [1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1e0, 1e1, 1e2, 1e3, 1e4])
    
    esn = EchoStateNetwork(
        n_input=d, n_reservoir=n_reservoir, 
        spectral_radius=spectral_radius, input_scaling=input_scaling,
        leak_factor=leak_factor, connectivity=connectivity, seed=base_seed
    )
    esn.fit(X_train, ridge_param)
    
    rmse_by_horizon = evaluate_esn_model(
        esn_model=esn, X_true_target=X_val_true, X_history=X_val,
        horizon_max=100, stride=10, h_list=horizons
    )
    return rmse_by_horizon[100]

# ============================================================
# 4. Automated Execution Workflow
# ============================================================
if __name__ == "__main__":
    print("Starting Automated Optuna ESN Parameter Search...\n")
    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=base_seed)
    )
    study.optimize(objective, n_trials=500, show_progress_bar=True)

    best_trial = study.best_trial
    best_params = best_trial.params

    print("\n" + "=" * 60)
    print("BEST ESN CONFIGURATION RETRIEVED")
    print("=" * 60)
    print(f"Reservoir Size  : {best_params['n_reservoir']}")
    print(f"Spectral Radius : {best_params['spectral_radius']:.4f}")
    print(f"Input Scaling   : {best_params['input_scaling']:.4f}")
    print(f"Leak Factor     : {best_params['leak_factor']:.4f}")
    print(f"Connectivity    : {best_params['connectivity']:.4f}")
    print(f"Ridge parameter : {best_params['ridge_param']:.6e}")
    print(f"Validation Target Average RMSE@100: {best_trial.value:.6f}")

    print("\n=== Launching Final Test Benchmark Using Best Values ===")
    
    num_runs = 25  # Ensemble verification runs for a model
    
    # Dictionary to aggregate RMSE values per horizon across all runs
    import numpy as np
    horizon_errors_all_runs = {h: [] for h in horizons}
    
    for run in range(num_runs):
        # Shifting the seed slightly per run ensures a distinct random reservoir topology
        run_seed = base_seed + run if base_seed is not None else None
        
        # Instantiate best configuration setup
        best_esn = EchoStateNetwork(
            n_input=d, 
            n_reservoir=best_params['n_reservoir'],
            spectral_radius=best_params['spectral_radius'],
            input_scaling=best_params['input_scaling'],
            leak_factor=best_params['leak_factor'],
            connectivity=best_params['connectivity'],
            seed=run_seed
        )
        best_esn.fit(X_train, best_params['ridge_param'])
        
        # Evaluate over the test block using identical overlapping stride setup
        rmse_by_horizon = evaluate_esn_model(
            esn_model=best_esn, X_true_target=X_test_true, X_history=X_test,
            horizon_max=100, stride=10, h_list=horizons
        )
        
        for h in horizons:
            horizon_errors_all_runs[h].append(rmse_by_horizon[h])

    # Compute mean and standard deviation across ensemble runs
    mean_test_rmses = {h: np.mean(horizon_errors_all_runs[h]) for h in horizons}
    std_test_rmses = {h: np.std(horizon_errors_all_runs[h]) for h in horizons}

    print("\n" + "=" * 60)
    print("FINAL ESN TEST EXTRAPOLATION SUMMARY")
    print("=" * 60)
    print(f"Parameters utilized: size={best_params['n_reservoir']}, rho={best_params['spectral_radius']:.3f}, leak={best_params['leak_factor']:.3f}")
    print("-" * 60)
    for h in horizons:
        print(f"Horizon {h}: RMSE = {mean_test_rmses[h]:.5f} ± {std_test_rmses[h]:.5f}")

Starting Automated Optuna ESN Parameter Search...



  0%|          | 0/500 [00:00<?, ?it/s]


BEST ESN CONFIGURATION RETRIEVED
Reservoir Size  : 512
Spectral Radius : 0.7916
Input Scaling   : 0.2246
Leak Factor     : 0.1619
Connectivity    : 0.0291
Ridge parameter : 1.000000e-03
Validation Target Average RMSE@100: 0.350291

=== Launching Final Test Benchmark Using Best Values ===

FINAL ESN TEST EXTRAPOLATION SUMMARY
Parameters utilized: size=512, rho=0.792, leak=0.162
------------------------------------------------------------
Horizon 25: RMSE = 0.21226 ± 0.02552
Horizon 50: RMSE = 0.34194 ± 0.05071
Horizon 75: RMSE = 0.44001 ± 0.06664
Horizon 100: RMSE = 0.52804 ± 0.06911
